In [2]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)


In [2]:
df = pd.read_csv('filtered.csv')
df.head()

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,il_bsarg_63,il_bsarg_64,il_bsarg_70,scenario,original_scenario,choice_made,lhw_h0,lhw_h1,lhw_h2,lhw_h3
0,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,0.00,0.00,0.00,h0,h2,0,0,7,42,54
1,870600,87060001,0,0,0,870600,87060001,32,1,0,...,469.93,469.93,469.93,h0,h2,0,0,11,44,50
2,870700,87070001,0,0,0,870700,87070001,61,0,0,...,1747.61,1747.61,1747.61,h0,h0,1,0,7,27,52
3,870900,87090001,0,0,0,870900,87090001,33,0,0,...,100.00,100.00,100.00,h0,h2,0,0,10,37,51
4,871700,87170002,0,0,87170001,871700,87170002,61,0,0,...,312.50,312.50,312.50,h0,h2,0,0,7,40,51


In [3]:
def map_lhw(row):
    # Ensure the scenario prefix 'h' is included when constructing the column name
    scenario_prefix = 'h' if not row["scenario"].startswith('h') else ''
    scenario_column_name = f'lhw_{scenario_prefix}{row["scenario"]}'
    return row[scenario_column_name]

# Apply the corrected function
df['lhw_scenario'] = df.apply(map_lhw, axis=1)

# Print a sample to verify the column has been created correctly
print(df[['idperson', 'scenario', 'lhw_scenario']].head())


   idperson scenario  lhw_scenario
0  87010001       h0             0
1  87060001       h0             0
2  87070001       h0             0
3  87090001       h0             0
4  87170002       h0             0


In [ ]:
def utility(c, l, beta1, beta2, gamma1, gamma2):
    epsilon = 1e-6  # Small constant to ensure c is strictly positive
    c_adjusted = np.maximum(c + epsilon, epsilon)  # Ensure c is positive
    l_adjusted = np.maximum(l, epsilon)  # Ensure l is positive, though l seems not to have this issue
    c_transformed = (np.power(c_adjusted, gamma1) - 1) / gamma1 if gamma1 != 0 else np.log(c_adjusted)
    l_transformed = (np.power(l_adjusted, gamma2) - 1) / gamma2 if gamma2 != 0 else np.log(l_adjusted)
    return beta1 * c_transformed + beta2 * l_transformed


In [ ]:
# Individual likelihood function using the log-sum-exp trick
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    beta1, beta2, gamma1, gamma2= params
    utilities = [
        utility(yds, 80 - lhw_actual, beta1, beta2, gamma1, gamma2),
        utility(yds, 80 - lhw_scenario, beta1, beta2, gamma1, gamma2)
    ]
    # Use log-sum-exp for numerical stability
    log_probabilities = utilities - logsumexp(utilities)
    # Return the negative log likelihood for the chosen scenario
    return -log_probabilities[int(choice_made)]

# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll




In [ ]:
def calculate_utilities(row, params):
    beta1, beta2, gamma1, gamma2 = params
    scenario = row['scenario']
    c = row['ils_udb_yds']  # Assuming this is your consumption proxy
    lhw = row[f'lhw_{scenario}']
    l = 80 - lhw  # Adjust based on your model's leisure definition
    u = utility(c, l, beta1, beta2, gamma1, gamma2)
    return u

In [ ]:
# Initial parameter guesses
initial_params = [0.365, 0.8031, 0.25, -0.002757]

# Minimize the total likelihood
result = minimize(total_likelihood, initial_params, args=(df,), method='BFGS')
print(result)

In [ ]:
# Define a wrapper function for the total likelihood that computes the gradient
def total_likelihood_with_grad(params, *args):
    eps = np.sqrt(np.finfo(float).eps)  # Suggested epsilon for numerical gradient
    grad = approx_fprime(params, total_likelihood, eps, *args)
    return total_likelihood(params, *args), grad

# Use the wrapper function in optimization
result = minimize(total_likelihood_with_grad, initial_params, args=(df,), method='Newton-CG', jac=True)
print(result)


In [ ]:
# Assuming df is your DataFrame and it's already loaded

# Example parameter values for exploration
exploration_params = [4.417e-01, -7.360e-07,  2.841e-01,  3.745e+00]

# Compute utilities for each row in the DataFrame using the exploration parameter values
df['computed_utility'] = df.apply(calculate_utilities, axis=1, args=(exploration_params,))

# Print the first few rows to see the computed utilities
print(df[['idperson', 'scenario', 'computed_utility']].head())


In [1]:
def jacobian(params, df):
    # Implementation of the analytical gradient of the total likelihood with respect to params
    # This requires deriving the partial derivatives of your likelihood function
    pass  # Replace with your gradient computation

# Then use it in the optimization
result = minimize(total_likelihood, initial_params, args=(df,), method='Newton-CG', jac=jacobian)
print(result)


NameError: name 'minimize' is not defined

In [ ]:
result = minimize(total_likelihood, initial_params, args=(df,), method='Nelder-Mead')
print(result)


In [4]:
### new utility 

def utility(c, l, alpha1, alpha2, beta1, beta2, gamma):
    """
    Quadratic utility function with interaction between c and l.

    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    """
    return alpha1 * c + alpha2 * c**2 + beta1 * l + beta2 * l**2 + gamma * c * l


In [5]:
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made):
    alpha1, alpha2, beta1, beta2, gamma = params  # Updated to match the new utility function
    utilities = [
        utility(yds, 80 - lhw_actual, alpha1, alpha2, beta1, beta2, gamma),
        utility(yds, 80 - lhw_scenario, alpha1, alpha2, beta1, beta2, gamma)
    ]
    # Continue with the log-sum-exp and return the negative log likelihood
    log_probabilities = utilities - logsumexp(utilities)
    return -log_probabilities[int(choice_made)]


In [6]:
# Total likelihood across all individuals
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        lhw_scenario = row[f'lhw_{row["scenario"]}']
        total_ll += ind_likelihood(
            params, 
            row['ils_udb_yds'], 
            row['lhw'],  # The actual lhw
            lhw_scenario,  # The lhw for the scenario
            row['choice_made']
        )
    return total_ll


In [8]:
# Adjust initial parameter guesses to match the updated utility function
initial_params = [0.365, 0.8031, 0.25, -0.002757, 0.1]  # Added an initial guess for the interaction term

# Run the optimization with the updated utility function
result = minimize(total_likelihood, initial_params, args=(df,), method='Nelder-Mead')
print(result)


       message: Optimization terminated successfully.
       success: True
        status: 0
           fun: 7938.00448662572
             x: [ 1.766e+00  9.197e-01  1.135e-01 -1.993e-03  4.341e-05]
           nit: 410
          nfev: 690
 final_simplex: (array([[ 1.766e+00,  9.197e-01, ..., -1.993e-03,
                         4.341e-05],
                       [ 1.766e+00,  9.197e-01, ..., -1.993e-03,
                         4.341e-05],
                       ...,
                       [ 1.766e+00,  9.197e-01, ..., -1.993e-03,
                         4.341e-05],
                       [ 1.765e+00,  9.197e-01, ..., -1.993e-03,
                         4.342e-05]]), array([ 7.938e+03,  7.938e+03,  7.938e+03,  7.938e+03,
                        7.938e+03,  7.938e+03]))


In [ ]:
from scipy.optimize import approx_fprime

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p, df), epsilon)
    return grad


In [ ]:
initial_params = [.766e+00,  9.197e-01,  1.135e-01, -1.993e-03,  4.341e-05] 
# Assume initial_params and df are already defined
result = minimize(total_likelihood, initial_params, args=(df,), method='Newton-CG', jac=jacobian)
print(result)